# Bài toán 3 — Đánh giá mô hình

## 7. Đánh giá và so sánh kết quả

### 7.1. Validation và chọn mô hình

MAE đo sai lệch tuyệt đối trung bình. RMSE nhấn mạnh các ngày sai nhiều hơn. R² so sánh tổng bình phương sai số với mức tham chiếu là trung bình thực tế trong giai đoạn chấm. MAE và RMSE trong bảng được tính theo triệu đơn vị tiền tệ mỗi ngày. Tiêu chí chọn đã đặt trước là MAE trung bình ba lượt. Thời gian tính trên cùng môi trường chỉ dùng để tham khảo.
    
Dùng dự báo đã tạo ở `02_mo_hinh.ipynb`. Các giá trị thực tế lấy từ `sales.csv` theo đúng ngày tương ứng.

In [2]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

sales_path = Path("data/raw/sales.csv")
start = Path(os.environ.get("RETAIL_ANALYTICS_ROOT", Path.cwd())).resolve()
project_root = next(
    (folder for folder in (start, *start.parents) if (folder / sales_path).is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw/sales.csv. Hãy đặt notebook trong dự án "
        "hoặc trỏ RETAIL_ANALYTICS_ROOT về thư mục gốc."
    )

prediction_path = project_root / "data/model/du_bao/forecast_predictions.csv"
if not prediction_path.is_file():
    raise FileNotFoundError("Chạy 02_mo_hinh.ipynb để tạo forecast_predictions.csv.")

predictions = pd.read_csv(
    prediction_path, parse_dates=["Date", "TrainEnd"], float_precision="round_trip",
)
targets = ["Revenue", "COGS"]

observed = pd.read_csv(
    project_root / sales_path, usecols=["Date", *targets], parse_dates=["Date"],
).set_index("Date").sort_index()[targets]
if not observed.index.is_unique:
    raise ValueError("sales.csv có ngày trùng. Kiểm tra lại ở 01_du_lieu.ipynb.")

contest_name = "Baseline cuộc thi"
fixed_name = "Mùa vụ, mức gốc cố định"
selected_label = "Phương pháp được chọn"


def prediction_frame(rows):
    return rows.set_index("Date")[targets].sort_index()


def test_forecast_for(method):
    return prediction_frame(predictions.loc[
        predictions["Split"].eq("test") & predictions["Method"].eq(method)
    ])


def evaluate(actual, forecast):
    pd.testing.assert_index_equal(actual.index, forecast.index)
    pd.testing.assert_index_equal(actual.columns, forecast.columns)
    if not np.isfinite(forecast.to_numpy(dtype=float)).all():
        raise ValueError("Dự báo có giá trị thiếu hoặc không hữu hạn.")
    actual = actual.astype(float) / 1_000_000
    error = forecast.astype(float) / 1_000_000 - actual
    squared_error = error.pow(2)
    variation = (actual - actual.mean()).pow(2).sum()
    return pd.DataFrame({
        "MAE": error.abs().mean(),
        "RMSE": np.sqrt(squared_error.mean()),
        "R²": 1 - squared_error.sum() / variation.where(variation > 0),
    }).rename_axis("Chỉ tiêu")

In [3]:
score_rows = []
validation = predictions.loc[predictions["Split"].eq("validation")]
for (fold_id, method), rows in validation.groupby(["Fold", "Method"], sort=False):
    forecast = prediction_frame(rows)
    score_rows.append(evaluate(observed.loc[forecast.index], forecast).assign(Fold=fold_id, Method=method))

validation_scores = pd.concat(score_rows).reset_index().rename(columns={"Method": "Phương pháp"})
summary = validation_scores.groupby(["Phương pháp", "Chỉ tiêu"])[["MAE", "RMSE", "R²"]].mean()
display(summary.unstack("Chỉ tiêu").reindex(validation["Method"].unique()).round(3))
selected_methods = summary["MAE"].unstack("Chỉ tiêu").idxmin().reindex(targets)
display(selected_methods.rename("Phương pháp được chọn"))

MAE           RMSE             R²        
Chỉ tiêu                      COGS Revenue   COGS Revenue   COGS Revenue
Phương pháp                                                             
Naive                        3.039   3.124  3.404   3.603 -3.967  -2.674
Trung bình 28 ngày           2.504   3.091  3.064   3.784 -2.268  -2.471
Theo thứ trong tuần          2.525   3.106  3.103   3.819 -2.357  -2.545
ETS (A,N,N)                  3.041   3.125  3.405   3.603 -3.968  -2.674
ARIMA(1,1,1)                 2.605   3.171  3.121   3.811 -2.683  -2.865
Hồi quy tuyến tính           2.138   2.460  2.510   2.905 -0.919  -0.827
Hồi quy tuyến tính + mùa vụ  1.617   1.858  1.910   2.228 -0.337  -0.285
Baseline cuộc thi            1.195   1.375  1.485   1.727  0.195   0.256
Mùa vụ, mức gốc cố định      1.048   1.212  1.324   1.548  0.374   0.414

Chỉ tiêu
Revenue    Mùa vụ, mức gốc cố định
COGS       Mùa vụ, mức gốc cố định
Name: Phương pháp được chọn, dtype: object

**Nhận xét.** Mùa vụ với mức gốc cố định có MAE trung bình thấp nhất cho cả Revenue và COGS, lần lượt **1,212** và **1,048** triệu đơn vị tiền tệ/ngày. Đây là phương pháp dùng để đánh giá cuối, cùng baseline cuộc thi làm mốc đối chiếu.

### 7.2. Đánh giá trên 548 ngày test

Tính lại các hệ số từ toàn bộ giai đoạn phát triển, đến hết 01/07/2021. Dự báo từ 02/07/2021 đến 31/12/2022. Giá trị thực tế trong tập test được đọc vào phép đánh giá sau khi tạo dự báo.

In [4]:
reference_test_forecast = test_forecast_for(contest_name)
selected_test_forecast = test_forecast_for(selected_label)
final_dates = selected_test_forecast.index
assert len(final_dates) == 548 and final_dates.is_unique
test_actual = observed.loc[final_dates]

test_metrics = pd.concat({
    contest_name: evaluate(test_actual, reference_test_forecast),
    selected_label: evaluate(test_actual, selected_test_forecast),
}, names=["Phương pháp"])
display(test_metrics.round(3))
mae_reduction = 100 * (1 - test_metrics.loc[selected_label, "MAE"] / test_metrics.loc[contest_name, "MAE"])
display(mae_reduction.rename("Giảm MAE (%)").to_frame().round(1))

MAE   RMSE     R²
Phương pháp           Chỉ tiêu                     
Baseline cuộc thi     Revenue   0.690  0.959  0.619
                      COGS      0.679  0.923  0.544
Phương pháp được chọn Revenue   0.596  0.821  0.721
                      COGS      0.542  0.752  0.697

,Giảm MAE (%)
Chỉ tiêu,
Revenue,13.7
COGS,20.2


### 7.3. Xuất bảng dự báo

Bật `EXPORT_FORECAST` để lưu 548 dòng dự báo cuối. Nếu đã có tệp, notebook kiểm tra nội dung trước khi xác nhận.

In [5]:
EXPORT_FORECAST = True
forecast_origin = pd.Timestamp(predictions.loc[predictions["Split"].eq("test"), "TrainEnd"].max())
output_path = project_root / "data/model/du_bao" / f"forecast_{forecast_origin:%Y-%m-%d}.csv"
forecast_csv = selected_test_forecast.rename_axis("Date").to_csv(
    date_format="%Y-%m-%d", float_format="%.2f", lineterminator=chr(10),
)

if EXPORT_FORECAST:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with output_path.open("x", encoding="utf-8", newline="") as file:
            file.write(forecast_csv)
    except FileExistsError:
        if output_path.read_text(encoding="utf-8") != forecast_csv:
            raise FileExistsError(f"Tệp đã tồn tại với nội dung khác: {output_path}")
    print(f"Đã xác nhận {len(selected_test_forecast)} dòng tại {output_path.relative_to(project_root)}")

display(selected_test_forecast.head())

Đã xác nhận 548 dòng tại data\model\du_bao\forecast_2021-07-01.csv


,Revenue,COGS
Date,,
2021-07-02,4609905.25,4365213.06
2021-07-03,4097775.31,3909822.32
2021-07-04,1283312.22,1152652.84
2021-07-05,1551951.80,1431980.12
2021-07-06,1667077.43,1576759.59
